# Snow Pole Detection - Model Evaluation

Comprehensive evaluation of trained YOLO models

**Required Metrics (Project Requirements):**
- Precision
- Recall
- mAP@50
- mAP@0.5:0.95
- Model size (for edge deployment)
- Inference speed (FPS)

**Note:** Test set has no labels, so we evaluate on validation set (standard practice)

## 1. Setup

In [1]:
import sys
import subprocess

def install_package(package_name, import_name=None):
    if import_name is None:
        import_name = package_name
    try:
        __import__(import_name)
        print(f"✓ {package_name}")
        return True
    except ImportError:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package_name, '--break-system-packages', '-q'])
        return True

print("Checking dependencies...")
install_package('torch')
install_package('ultralytics')

Checking dependencies...
✓ torch
✓ ultralytics


True

In [3]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import numpy as np
from datetime import datetime

import torch
from ultralytics import YOLO

print(f"\n✓ All packages loaded")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")


✓ All packages loaded
PyTorch: 2.9.1+cu128
CUDA: True
GPU: NVIDIA GeForce RTX 4090


## 2. Configuration

Uses the local `training_data/` directory created by Training notebook.

In [4]:
# ============================================================
# CONFIGURATION
# ============================================================

# Local training data directory (created by Training notebook)
WORK_DIR = Path('training_data')
DATA_YAML = WORK_DIR / 'data.yaml'

# Project name (same as training)
PROJECT_NAME = "snowpole_detection"
RUNS_DIR = Path(f"runs/{PROJECT_NAME}")

# Output directory for evaluation results
EVAL_DIR = RUNS_DIR / 'evaluation'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# Check if data exists
print(f"Data YAML: {DATA_YAML}")
print(f"Data YAML exists: {DATA_YAML.exists()}")
print(f"Runs directory: {RUNS_DIR}")
print(f"Evaluation output: {EVAL_DIR}")

if not DATA_YAML.exists():
    print("\n⚠️  data.yaml not found!")
    print("Please run Training.ipynb first to prepare the dataset.")
else:
    # Show dataset info
    import yaml
    with open(DATA_YAML, 'r') as f:
        data_config = yaml.safe_load(f)
    
    print(f"\nDataset config:")
    print(f"  Path: {data_config.get('path', 'N/A')}")
    print(f"  Classes: {data_config.get('names', [])}")
    
    # Count images
    for split in ['train', 'val', 'test']:
        img_dir = WORK_DIR / split / 'images'
        lbl_dir = WORK_DIR / split / 'labels'
        if img_dir.exists():
            n_imgs = len(list(img_dir.glob('*')))
            n_lbls = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
            print(f"  {split}: {n_imgs} images, {n_lbls} labels")

Data YAML: training_data/data.yaml
Data YAML exists: True
Runs directory: runs/snowpole_detection
Evaluation output: runs/snowpole_detection/evaluation

Dataset config:
  Path: /home/maryasae/Downloads/files/training_data
  Classes: ['pole']
  train: 1264 images, 1264 labels
  val: 353 images, 353 labels
  test: 184 images, 0 labels


## 3. Find Trained Models

In [5]:
def find_trained_models(runs_dir):
    """Find all trained model weights in runs directory."""
    models = []
    
    if not runs_dir.exists():
        print(f"⚠️  Runs directory not found: {runs_dir}")
        return models
    
    for exp_dir in runs_dir.iterdir():
        if not exp_dir.is_dir():
            continue
        
        # Skip evaluation directory
        if exp_dir.name == 'evaluation':
            continue
        
        best_weights = exp_dir / 'weights' / 'best.pt'
        
        if best_weights.exists():
            # Load metadata if exists
            metadata = {}
            metadata_file = exp_dir / 'training_metadata.json'
            if metadata_file.exists():
                with open(metadata_file, 'r') as f:
                    metadata = json.load(f)
            
            models.append({
                'name': exp_dir.name,
                'path': best_weights,
                'experiment_dir': exp_dir,
                'model_type': metadata.get('model', exp_dir.name.split('_')[0]),
                'metadata': metadata
            })
    
    return models

trained_models = find_trained_models(RUNS_DIR)

print(f"\nFound {len(trained_models)} trained model(s):")
for i, model in enumerate(trained_models, 1):
    print(f"  {i}. {model['model_type']}: {model['name']}")

if len(trained_models) == 0:
    print("\n⚠️  No trained models found!")
    print("Please run Training.ipynb first.")


Found 3 trained model(s):
  1. yolov8n: yolov8n_20251122_112324
  2. yolov9t: yolov9t_20251122_113325
  3. yolo11n: yolo11n_20251122_115057


## 4. Evaluation Function

In [6]:
def evaluate_model(model_path, data_yaml, device='cpu'):
    """
    Comprehensive model evaluation.
    Returns all required metrics for project report.
    """
    print(f"\nLoading model: {model_path.name}")
    model = YOLO(str(model_path))
    
    # Model size
    model_size_mb = model_path.stat().st_size / (1024 * 1024)
    
    # Count parameters
    try:
        total_params = sum(p.numel() for p in model.model.parameters())
        trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
    except:
        total_params = 0
        trainable_params = 0
    
    print(f"  Model size: {model_size_mb:.2f} MB")
    print(f"  Parameters: {total_params:,}")
    
    # Run validation
    print(f"\n  Running validation...")
    start_time = time.time()
    
    metrics = model.val(
        data=str(data_yaml),
        split='val',
        batch=16,
        device=device,
        verbose=False,
        plots=True,
        save_json=True,
    )
    
    eval_time = time.time() - start_time
    
    # Extract metrics
    results = {
        'model_path': str(model_path),
        'model_size_mb': model_size_mb,
        'total_params': total_params,
        'trainable_params': trainable_params,
        'eval_time_seconds': eval_time,
        
        # Required metrics
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'mAP50': float(metrics.box.map50),
        'mAP50_95': float(metrics.box.map),
        
        # F1 score
        'f1_score': 2 * (float(metrics.box.mp) * float(metrics.box.mr)) / 
                    (float(metrics.box.mp) + float(metrics.box.mr)) 
                    if (float(metrics.box.mp) + float(metrics.box.mr)) > 0 else 0,
        
        # Speed metrics
        'speed_preprocess_ms': float(metrics.speed.get('preprocess', 0)),
        'speed_inference_ms': float(metrics.speed.get('inference', 0)),
        'speed_postprocess_ms': float(metrics.speed.get('postprocess', 0)),
    }
    
    results['speed_total_ms'] = (results['speed_preprocess_ms'] + 
                                  results['speed_inference_ms'] + 
                                  results['speed_postprocess_ms'])
    results['fps'] = 1000 / results['speed_total_ms'] if results['speed_total_ms'] > 0 else 0
    
    # Display
    print(f"\n  {'='*50}")
    print(f"  RESULTS")
    print(f"  {'='*50}")
    print(f"  Precision:      {results['precision']:.4f}")
    print(f"  Recall:         {results['recall']:.4f}")
    print(f"  F1-Score:       {results['f1_score']:.4f}")
    print(f"  mAP@50:         {results['mAP50']:.4f}")
    print(f"  mAP@0.5:0.95:   {results['mAP50_95']:.4f}")
    print(f"  ")
    print(f"  Size:           {results['model_size_mb']:.2f} MB")
    print(f"  Inference:      {results['speed_inference_ms']:.2f} ms")
    print(f"  FPS:            {results['fps']:.1f}")
    print(f"  {'='*50}")
    
    return results

print("✓ Evaluation function ready")

✓ Evaluation function ready


## 5. Evaluate All Models

In [7]:
all_results = []

if len(trained_models) == 0:
    print("⚠️  No models to evaluate!")
else:
    device = 0 if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    for i, model_info in enumerate(trained_models, 1):
        print(f"\n{'='*80}")
        print(f"Evaluating {i}/{len(trained_models)}: {model_info['model_type']}")
        print(f"{'='*80}")
        
        try:
            results = evaluate_model(
                model_path=model_info['path'],
                data_yaml=DATA_YAML,
                device=device
            )
            
            results['model_name'] = model_info['name']
            results['model_type'] = model_info['model_type']
            results['experiment_dir'] = str(model_info['experiment_dir'])
            
            # Save individual results
            results_file = model_info['experiment_dir'] / 'evaluation_results.json'
            with open(results_file, 'w') as f:
                json.dump(results, f, indent=2)
            print(f"\n  ✓ Saved: {results_file}")
            
            all_results.append(results)
            
        except Exception as e:
            print(f"\n  ✗ Error: {e}")
            import traceback
            traceback.print_exc()
    
    print(f"\n{'='*80}")
    print(f"✓ Evaluated {len(all_results)}/{len(trained_models)} models")
    print(f"{'='*80}")

Using device: 0

Evaluating 1/3: yolov8n

Loading model: best.pt
  Model size: 5.95 MB
  Parameters: 3,011,043

  Running validation...
Ultralytics 8.3.229 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24058MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6740.7±4006.4 MB/s, size: 1094.0 KB)
val: Scanning /home/maryasae/Downloads/files/training_data/val/labels.cache... 353 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 353/353 1.1Mit/s 0.0s0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 22.2it/s 1.0s0.1s
                   all        353        443      0.889      0.779      0.886      0.695
Speed: 0.4ms preprocess, 0.7ms inference, 0.0ms loss, 0.3ms postprocess per image
Saving /home/maryasae/Downloads/files/runs/detect/val/predictions.json...
Results saved to /home/maryasae/Downloads/files/runs/detect/val


## 6. Results Comparison

In [8]:
if len(all_results) > 0:
    df = pd.DataFrame([{
        'Model': r['model_type'],
        'Precision': f"{r['precision']:.4f}",
        'Recall': f"{r['recall']:.4f}",
        'F1': f"{r['f1_score']:.4f}",
        'mAP@50': f"{r['mAP50']:.4f}",
        'mAP@50-95': f"{r['mAP50_95']:.4f}",
        'Size(MB)': f"{r['model_size_mb']:.2f}",
        'Params(M)': f"{r['total_params']/1e6:.2f}",
        'Inf(ms)': f"{r['speed_inference_ms']:.2f}",
        'FPS': f"{r['fps']:.1f}"
    } for r in all_results])
    
    print(f"\n{'='*100}")
    print("MODEL COMPARISON")
    print(f"{'='*100}")
    print(df.to_string(index=False))
    print(f"{'='*100}")
    
    # Save
    df.to_csv(EVAL_DIR / 'comparison.csv', index=False)
    print(f"\n✓ Saved: {EVAL_DIR / 'comparison.csv'}")
    
    with open(EVAL_DIR / 'all_results.json', 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f"✓ Saved: {EVAL_DIR / 'all_results.json'}")


MODEL COMPARISON
  Model Precision Recall     F1 mAP@50 mAP@50-95 Size(MB) Params(M) Inf(ms)   FPS
yolov8n    0.8885 0.7788 0.8300 0.8859    0.6953     5.95      3.01    0.75 690.7
yolov9t    0.9183 0.7607 0.8321 0.8784    0.6789     4.41      2.01    0.73 684.7
yolo11n    0.9204 0.8397 0.8782 0.9168    0.7005     5.21      2.59    0.45 840.9

✓ Saved: runs/snowpole_detection/evaluation/comparison.csv
✓ Saved: runs/snowpole_detection/evaluation/all_results.json


## 7. Visualization

In [9]:
if len(all_results) >= 1:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    models = [r['model_type'] for r in all_results]
    colors = sns.color_palette("husl", len(models))
    
    # 1. Main metrics
    metrics = ['precision', 'recall', 'mAP50', 'mAP50_95']
    metric_labels = ['Precision', 'Recall', 'mAP@50', 'mAP@50-95']
    x = np.arange(len(models))
    width = 0.2
    
    for i, (metric, label) in enumerate(zip(metrics, metric_labels)):
        values = [r[metric] for r in all_results]
        axes[0, 0].bar(x + i*width, values, width, label=label, alpha=0.8)
    
    axes[0, 0].set_ylabel('Score')
    axes[0, 0].set_title('Detection Metrics')
    axes[0, 0].set_xticks(x + width*1.5)
    axes[0, 0].set_xticklabels(models)
    axes[0, 0].legend()
    axes[0, 0].set_ylim(0, 1)
    axes[0, 0].grid(axis='y', alpha=0.3)
    
    # 2. Speed comparison
    fps_values = [r['fps'] for r in all_results]
    axes[0, 1].bar(models, fps_values, color=colors, alpha=0.8)
    axes[0, 1].set_ylabel('FPS')
    axes[0, 1].set_title('Inference Speed')
    axes[0, 1].grid(axis='y', alpha=0.3)
    for i, v in enumerate(fps_values):
        axes[0, 1].text(i, v + 1, f'{v:.1f}', ha='center')
    
    # 3. Model size
    sizes = [r['model_size_mb'] for r in all_results]
    axes[1, 0].bar(models, sizes, color=colors, alpha=0.8)
    axes[1, 0].set_ylabel('Size (MB)')
    axes[1, 0].set_title('Model Size')
    axes[1, 0].grid(axis='y', alpha=0.3)
    for i, v in enumerate(sizes):
        axes[1, 0].text(i, v + 0.1, f'{v:.1f}', ha='center')
    
    # 4. Accuracy vs Speed
    for i, r in enumerate(all_results):
        axes[1, 1].scatter(r['fps'], r['mAP50_95'], s=r['model_size_mb']*20, 
                           c=[colors[i]], alpha=0.7, label=r['model_type'])
    axes[1, 1].set_xlabel('FPS')
    axes[1, 1].set_ylabel('mAP@50-95')
    axes[1, 1].set_title('Accuracy vs Speed (bubble size = model size)')
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)
    
    plt.tight_layout()
    
    plot_path = EVAL_DIR / 'comparison_plots.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n✓ Saved: {plot_path}")

<Figure size 1400x1000 with 4 Axes>


✓ Saved: runs/snowpole_detection/evaluation/comparison_plots.png


## 8. Model Recommendation

In [10]:
if len(all_results) > 0:
    # Calculate efficiency score: mAP * FPS / size
    for r in all_results:
        r['efficiency_score'] = (r['mAP50_95'] * r['fps']) / r['model_size_mb']
    
    best_accuracy = max(all_results, key=lambda x: x['mAP50_95'])
    best_speed = max(all_results, key=lambda x: x['fps'])
    best_size = min(all_results, key=lambda x: x['model_size_mb'])
    best_balance = max(all_results, key=lambda x: x['efficiency_score'])
    
    print(f"\n{'='*80}")
    print("MODEL RECOMMENDATION")
    print(f"{'='*80}")
    
    print(f"\n🎯 Best Accuracy: {best_accuracy['model_type']}")
    print(f"   mAP@50-95: {best_accuracy['mAP50_95']:.4f}")
    
    print(f"\n⚡ Fastest: {best_speed['model_type']}")
    print(f"   FPS: {best_speed['fps']:.1f}")
    
    print(f"\n📦 Smallest: {best_size['model_type']}")
    print(f"   Size: {best_size['model_size_mb']:.2f} MB")
    
    print(f"\n⚖️  Best Balance (Recommended): {best_balance['model_type']}")
    print(f"   mAP@50-95: {best_balance['mAP50_95']:.4f}")
    print(f"   FPS: {best_balance['fps']:.1f}")
    print(f"   Size: {best_balance['model_size_mb']:.2f} MB")
    
    print(f"\n{'='*80}")
    print(f"💡 RECOMMENDATION FOR EDGE DEPLOYMENT:")
    print(f"   Use: {best_balance['model_type']}")
    print(f"   Path: {best_balance['model_path']}")
    print(f"{'='*80}")
    
    # Save recommendation
    recommendation = {
        'recommended': best_balance['model_type'],
        'reason': 'Best balance of accuracy, speed, and size',
        'metrics': {
            'precision': best_balance['precision'],
            'recall': best_balance['recall'],
            'mAP50': best_balance['mAP50'],
            'mAP50_95': best_balance['mAP50_95'],
            'fps': best_balance['fps'],
            'size_mb': best_balance['model_size_mb']
        },
        'model_path': best_balance['model_path'],
        'alternatives': {
            'best_accuracy': best_accuracy['model_type'],
            'fastest': best_speed['model_type'],
            'smallest': best_size['model_type']
        }
    }
    
    with open(EVAL_DIR / 'recommendation.json', 'w') as f:
        json.dump(recommendation, f, indent=2)
    print(f"\n✓ Saved: {EVAL_DIR / 'recommendation.json'}")


MODEL RECOMMENDATION

🎯 Best Accuracy: yolo11n
   mAP@50-95: 0.7005

⚡ Fastest: yolo11n
   FPS: 840.9

📦 Smallest: yolov9t
   Size: 4.41 MB

⚖️  Best Balance (Recommended): yolo11n
   mAP@50-95: 0.7005
   FPS: 840.9
   Size: 5.21 MB

💡 RECOMMENDATION FOR EDGE DEPLOYMENT:
   Use: yolo11n
   Path: runs/snowpole_detection/yolo11n_20251122_115057/weights/best.pt

✓ Saved: runs/snowpole_detection/evaluation/recommendation.json


## 9. Sustainability Analysis

In [11]:
if len(all_results) > 0:
    print(f"\n{'='*80}")
    print("SUSTAINABILITY ANALYSIS")
    print(f"{'='*80}")
    
    # Power estimates
    GPU_POWER = {'RTX 4090': 450, 'RTX 3090': 350, 'CPU': 150}
    TESLA_WH_PER_KM = 150  # Tesla Model Y
    TRONDHEIM_OSLO_KM = 500
    
    total_training_hours = 0
    
    for model_info in trained_models:
        metadata = model_info.get('metadata', {})
        hours = metadata.get('training_time_hours', 0)
        total_training_hours += hours
        
        if hours > 0:
            # Assume RTX 4090 for Cybele lab
            power_watts = GPU_POWER['RTX 4090']
            energy_kwh = (power_watts * hours) / 1000
            tesla_km = (energy_kwh * 1000) / TESLA_WH_PER_KM
            
            print(f"\n{model_info['model_type']}:")
            print(f"  Training time: {hours:.2f} hours")
            print(f"  Energy: {energy_kwh:.2f} kWh")
            print(f"  Tesla equivalent: {tesla_km:.1f} km ({tesla_km/TRONDHEIM_OSLO_KM*100:.1f}% of Trondheim→Oslo)")
    
    if total_training_hours > 0:
        total_energy = (GPU_POWER['RTX 4090'] * total_training_hours) / 1000
        total_tesla = (total_energy * 1000) / TESLA_WH_PER_KM
        
        print(f"\n{'='*80}")
        print(f"TOTAL:")
        print(f"  Training time: {total_training_hours:.2f} hours")
        print(f"  Energy: {total_energy:.2f} kWh")
        print(f"  Tesla equivalent: {total_tesla:.1f} km")
        print(f"{'='*80}")
        
        # Save sustainability report
        sustainability = {
            'total_training_hours': total_training_hours,
            'assumed_gpu': 'RTX 4090',
            'gpu_power_watts': GPU_POWER['RTX 4090'],
            'total_energy_kwh': total_energy,
            'tesla_model_y_equivalent_km': total_tesla,
            'trondheim_oslo_percentage': (total_tesla / TRONDHEIM_OSLO_KM) * 100
        }
        
        with open(EVAL_DIR / 'sustainability.json', 'w') as f:
            json.dump(sustainability, f, indent=2)
        print(f"\n✓ Saved: {EVAL_DIR / 'sustainability.json'}")


SUSTAINABILITY ANALYSIS

yolov8n:
  Training time: 0.17 hours
  Energy: 0.08 kWh
  Tesla equivalent: 0.5 km (0.1% of Trondheim→Oslo)

yolov9t:
  Training time: 0.29 hours
  Energy: 0.13 kWh
  Tesla equivalent: 0.9 km (0.2% of Trondheim→Oslo)

yolo11n:
  Training time: 0.18 hours
  Energy: 0.08 kWh
  Tesla equivalent: 0.5 km (0.1% of Trondheim→Oslo)

TOTAL:
  Training time: 0.64 hours
  Energy: 0.29 kWh
  Tesla equivalent: 1.9 km

✓ Saved: runs/snowpole_detection/evaluation/sustainability.json


## 10. Report Summary

In [12]:
if len(all_results) > 0:
    print(f"\n{'='*80}")
    print("REPORT SUMMARY (for project presentation)")
    print(f"{'='*80}")
    
    print(f"\n## Evaluation Results")
    print(f"\n### Dataset")
    print(f"- Evaluated on: Validation set")
    val_count = len(list((WORK_DIR / 'val' / 'images').glob('*'))) if (WORK_DIR / 'val' / 'images').exists() else 0
    print(f"- Images: {val_count}")
    
    print(f"\n### Model Performance")
    for r in all_results:
        print(f"\n**{r['model_type']}:**")
        print(f"- Precision: {r['precision']:.4f}")
        print(f"- Recall: {r['recall']:.4f}")
        print(f"- mAP@50: {r['mAP50']:.4f}")
        print(f"- mAP@0.5:0.95: {r['mAP50_95']:.4f}")
        print(f"- Model Size: {r['model_size_mb']:.2f} MB")
        print(f"- Inference: {r['speed_inference_ms']:.2f} ms ({r['fps']:.1f} FPS)")
    
    print(f"\n### Recommendation")
    print(f"**{best_balance['model_type']}** - Best for edge deployment")
    print(f"- Achieves {best_balance['mAP50_95']:.4f} mAP@0.5:0.95")
    print(f"- Runs at {best_balance['fps']:.1f} FPS")
    print(f"- Size: {best_balance['model_size_mb']:.2f} MB")
    
    print(f"\n{'='*80}")


REPORT SUMMARY (for project presentation)

## Evaluation Results

### Dataset
- Evaluated on: Validation set
- Images: 353

### Model Performance

**yolov8n:**
- Precision: 0.8885
- Recall: 0.7788
- mAP@50: 0.8859
- mAP@0.5:0.95: 0.6953
- Model Size: 5.95 MB
- Inference: 0.75 ms (690.7 FPS)

**yolov9t:**
- Precision: 0.9183
- Recall: 0.7607
- mAP@50: 0.8784
- mAP@0.5:0.95: 0.6789
- Model Size: 4.41 MB
- Inference: 0.73 ms (684.7 FPS)

**yolo11n:**
- Precision: 0.9204
- Recall: 0.8397
- mAP@50: 0.9168
- mAP@0.5:0.95: 0.7005
- Model Size: 5.21 MB
- Inference: 0.45 ms (840.9 FPS)

### Recommendation
**yolo11n** - Best for edge deployment
- Achieves 0.7005 mAP@0.5:0.95
- Runs at 840.9 FPS
- Size: 5.21 MB



In [13]:
print(f"\n{'='*80}")
print("EVALUATION COMPLETE")
print(f"{'='*80}")
print(f"\nOutput files in: {EVAL_DIR}/")
print(f"  - comparison.csv")
print(f"  - comparison_plots.png")
print(f"  - all_results.json")
print(f"  - recommendation.json")
print(f"  - sustainability.json")
print(f"\n✅ Ready for project report!")


EVALUATION COMPLETE

Output files in: runs/snowpole_detection/evaluation/
  - comparison.csv
  - comparison_plots.png
  - all_results.json
  - recommendation.json
  - sustainability.json

✅ Ready for project report!
